# Gait classification from skeleton data

This notebook follows a simple pipeline:

1. Load and validate each Parquet file
2. Build one feature row per walk
3. Compare several classifiers using grouped cross-validation
4. Save the best model and evaluation outputs
5. Predict a single walk from the saved model

The project keeps the pipeline explicit and modular so you can run or troubleshoot each stage independently.

In [7]:
# Part 1: Imports, constants, and configuration
# Data and output paths are fixed here so the workflow stays reproducible.
from pathlib import Path
import logging
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
PARQUET_DIR = Path('../data/parquet')
FEATURES_PATH = Path('../data/features.csv')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
LOGGER = logging.getLogger('gait_notebook')

GAITS = ['normal', 'antalgic', 'stiff_legged', 'lurch', 'steppage', 'trendelenburg']
REQUIRED_METADATA = ['subject_id', 'gait_label', 'walk_id', 'source_file', 'n_frames']
EXPECTED_COLUMNS = ['subject_id', 'gait_label', 'walk_id', 'sensor_id', 'frame_index', 'source_file', 'n_frames', 'time'] + [f'joint{j}_{a}' for j in range(25) for a in ('x', 'y', 'z')]
JOINTS = {'SpineBase': 0, 'SpineShoulder': 20, 'HipLeft': 12, 'KneeLeft': 13, 'AnkleLeft': 14, 'HipRight': 16, 'KneeRight': 17, 'AnkleRight': 18}
COORDINATE_COLUMNS = [f'joint{j}_{a}' for j in range(25) for a in ('x', 'y', 'z')]


## 1) Load and validate the Parquet data

This section confirms each file exists and that all frames use the expected schema before any modeling begins.

In [8]:
# Part 2: Data loading and schema validation
# Fail early if files are missing or the schema changes unexpectedly.
def load_parquet_frames():
    frames = []
    for gait in GAITS:
        path = PARQUET_DIR / f'{gait}.parquet'
        if not path.exists():
            raise FileNotFoundError(f'Missing Parquet file: {path}')

        frame = pd.read_parquet(path)
        if list(frame.columns) != EXPECTED_COLUMNS:
            raise ValueError(f'Unexpected columns in {path.name}; got {list(frame.columns)}')
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True)
    LOGGER.info('Loaded %d frames from %d Parquet files', len(combined), len(frames))
    return combined

frames = load_parquet_frames()
print(f'Loaded {len(frames)} total rows across all gait files.')

# Hold out 2 subjects as a true test set, before any feature engineering or model selection
TEST_SUBJECTS = sorted(frames['subject_id'].astype(str).unique())[-2:]
test_frames = frames[frames['subject_id'].astype(str).isin(TEST_SUBJECTS)]
train_frames = frames[~frames['subject_id'].astype(str).isin(TEST_SUBJECTS)]
print('Test subjects (held out):', TEST_SUBJECTS)
print(f'Train rows: {len(train_frames)}, test rows: {len(test_frames)}')


INFO: Loaded 340895 frames from 6 Parquet files


Loaded 340895 total rows across all gait files.
Test subjects (held out): ['8', '9']
Train rows: 275310, test rows: 65585


## 2) Feature engineering for each walk

Each walk is converted into a single feature vector using joint angles, trunk lean, pelvis position, and movement speed. This keeps the learning step focused on a compact, interpretable representation.

In [9]:
# Part 3A: Joint coordinate helpers
# These helpers convert joint IDs into 3D coordinates and compute knee angles.
def point(frame, joint):
    joint_index = JOINTS[joint]
    return frame[[f'joint{joint_index}_x', f'joint{joint_index}_y', f'joint{joint_index}_z']].to_numpy(dtype=float)


def knee_angle(frame, hip, knee, ankle):
    a = point(frame, hip) - point(frame, knee)
    b = point(frame, ankle) - point(frame, knee)
    denominator = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    # Guard against zero-length vectors; they would otherwise divide by zero.
    cosine = np.divide(
        (a * b).sum(axis=1),
        denominator,
        out=np.full(len(frame), np.nan),
        where=denominator > 0,
    )
    return np.degrees(np.arccos(np.clip(cosine, -1, 1)))


In [10]:
# Part 3B: Build one feature vector from one walk
# Each walk becomes a single row in the training table.
def feature_row(group):
    group = group.sort_values(['frame_index', 'sensor_id']).reset_index(drop=True)
    left = knee_angle(group, 'HipLeft', 'KneeLeft', 'AnkleLeft')
    right = knee_angle(group, 'HipRight', 'KneeRight', 'AnkleRight')
    spine = point(group, 'SpineShoulder') - point(group, 'SpineBase')
    spine_norm = np.linalg.norm(spine, axis=1)
    # Trunk lean is measured from the vertical axis of the spine vector.
    trunk = np.degrees(
        np.arccos(
            np.divide(
                spine[:, 1],
                spine_norm,
                out=np.full(len(group), np.nan),
                where=spine_norm > 0,
            )
        )
    )
    base = point(group, 'SpineBase')
    time = group['time'].to_numpy(dtype=float)
    diffs = np.diff(np.sort(np.unique(time)))
    fps = 1 / np.median(diffs) if len(diffs) and np.median(diffs) > 0 else np.nan
    duration = len(group) / fps if np.isfinite(fps) and fps > 0 else np.nan
    distance = np.sqrt(np.diff(base[:, 0]) ** 2 + np.diff(base[:, 2]) ** 2).sum()
    return {
        'subject_id': str(group['subject_id'].iloc[0]),
        'source_file': str(group['source_file'].iloc[0]),
        'gait_label': str(group['gait_label'].iloc[0]),
        'walk_id': int(group['walk_id'].iloc[0]),
        'left_knee_angle_mean': np.nanmean(left),
        'left_knee_angle_min': np.nanmin(left),
        'left_knee_angle_max': np.nanmax(left),
        'left_knee_angle_std': np.nanstd(left),
        'right_knee_angle_mean': np.nanmean(right),
        'right_knee_angle_min': np.nanmin(right),
        'right_knee_angle_max': np.nanmax(right),
        'right_knee_angle_std': np.nanstd(right),
        'trunk_lean_mean': np.nanmean(trunk),
        'trunk_lean_std': np.nanstd(trunk),
        'pelvis_drop_mean': np.nanmean(group['joint12_y'] - group['joint16_y']),
        'pelvis_drop_std': np.nanstd(group['joint12_y'] - group['joint16_y']),
        'sway_std': np.nanstd(base[:, 0]),
        'speed': distance / duration if np.isfinite(duration) and duration > 0 else np.nan,
        'knee_range_symmetry': abs((np.nanmax(left) - np.nanmin(left)) - (np.nanmax(right) - np.nanmin(right))),
        'ankle_height_range_symmetry': abs((group['joint14_y'].max() - group['joint14_y'].min()) - (group['joint18_y'].max() - group['joint18_y'].min())),
    }


In [11]:
# Part 3C: Filter bad walks and build the final feature table
# Invalid or noisy walks are skipped before modeling so the classifier learns from clean data.
def extract_features(frame):
    rows, dropped = [], {'nan': 0, 'out_of_range': 0, 'feature_error': 0}
    for (_, _), group in frame.groupby(['subject_id', 'source_file'], sort=True):
        if group[COORDINATE_COLUMNS].abs().gt(10).any().any():
            dropped['out_of_range'] += 1
            continue
        if group[COORDINATE_COLUMNS].isna().any().any():
            dropped['nan'] += 1
            continue
        try:
            row = feature_row(group)
            numeric = pd.Series(row).drop(labels=['subject_id', 'source_file', 'gait_label', 'walk_id']).to_numpy(dtype=float)
            if not np.isfinite(numeric).all():
                dropped['nan'] += 1
                continue
            rows.append(row)
        except (ValueError, KeyError, FloatingPointError):
            dropped['feature_error'] += 1

    result = pd.DataFrame(rows)
    if result.empty:
        raise ValueError('No valid walks remain after cleaning')
    return result, dropped

features, dropped = extract_features(train_frames)
test_features, _ = extract_features(test_frames)
features.to_csv(FEATURES_PATH, index=False)
print('Dropped walks:', dropped, 'total=', sum(dropped.values()))
print('Feature shape:', features.shape)
print('Walks per class:')
print(features['gait_label'].value_counts().sort_index())

test_features.to_csv(Path('../data/test_features.csv'), index=False)


Dropped walks: {'nan': 0, 'out_of_range': 49, 'feature_error': 0} total= 49
Feature shape: (2182, 20)
Walks per class:
gait_label
antalgic         366
lurch            359
normal           364
steppage         370
stiff_legged     366
trendelenburg    357
Name: count, dtype: int64


## 3) Compare candidate models

We train several simple models using subject-grouped cross-validation so that all walks from the same subject stay in the same fold.

In [12]:
# Part 4: Model training and cross-validation
# Grouped folds keep all walks from the same subject inside one split.
def make_models():
    return {
        'LogisticRegression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))]),
        'RandomForest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
        'GradientBoosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
        'Dummy': DummyClassifier(strategy='most_frequent'),
    }

id_columns = ['subject_id', 'source_file', 'gait_label', 'walk_id']
feature_columns = [c for c in features.columns if c not in id_columns]
X, y, groups = features[feature_columns], features['gait_label'], features['subject_id']
splitter = GroupKFold(n_splits=5)
records = []

for name, model in make_models().items():
    accuracy_scores, f1_scores = [], []
    for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups), start=1):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        predicted = model.predict(X.iloc[test_idx])
        accuracy_scores.append(accuracy_score(y.iloc[test_idx], predicted))
        f1_scores.append(f1_score(y.iloc[test_idx], predicted, average='macro'))
        records.append({
            'model': name,
            'fold': fold,
            'accuracy': accuracy_scores[-1],
            'macro_f1': f1_scores[-1],
        })

    records.extend([
        {'model': name, 'fold': 'mean', 'accuracy': np.mean(accuracy_scores), 'macro_f1': np.mean(f1_scores)},
        {'model': name, 'fold': 'std', 'accuracy': np.std(accuracy_scores), 'macro_f1': np.std(f1_scores)},
    ])

fold_table = pd.DataFrame(records)
summary = fold_table[fold_table['fold'].isin(['mean', 'std'])].pivot(index='model', columns='fold', values=['accuracy', 'macro_f1'])
summary.columns = [f'{metric}_{stat}' for metric, stat in summary.columns]
comparison = summary.reset_index().sort_values('macro_f1_mean', ascending=False)

fold_table.to_csv(OUTPUT_DIR / 'model_comparison_folds.csv', index=False)
comparison.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)

print('Model comparison:')
print(comparison.to_string(index=False))


Model comparison:
             model  accuracy_mean  accuracy_std  macro_f1_mean  macro_f1_std
LogisticRegression       0.756248      0.066095       0.747040      0.075106
  GradientBoosting       0.740752      0.083871       0.733268      0.084656
      RandomForest       0.736788      0.053733       0.726978      0.056397
             Dummy       0.167780      0.003614       0.047889      0.000883


## 4) Select the best model and save diagnostics

We exclude the dummy baseline from the model choice, then evaluate the winning classifier on every validation fold and save the confusion matrix and prediction table.

In [13]:
# Part 5: Best-model selection and artifact export
candidate_summary = comparison[comparison['model'] != 'Dummy']
best_name = candidate_summary.iloc[0]['model']
best_model = make_models()[best_name]
labels = sorted(y.unique())
all_true, all_pred, prediction_rows = [], [], []

for train_idx, test_idx in splitter.split(X, y, groups):
    best_model.fit(X.iloc[train_idx], y.iloc[train_idx])
    predicted = best_model.predict(X.iloc[test_idx])
    probabilities = best_model.predict_proba(X.iloc[test_idx])
    all_true.extend(y.iloc[test_idx])
    all_pred.extend(predicted)

    for local, row_index in enumerate(test_idx):
        row = {
            'subject_id': features.iloc[row_index]['subject_id'],
            'source_file': features.iloc[row_index]['source_file'],
            'true_label': y.iloc[row_index],
            'predicted_label': predicted[local],
        }
        row.update({f'prob_{label}': probabilities[local, list(best_model.classes_).index(label)] for label in labels})
        prediction_rows.append(row)

matrix = confusion_matrix(all_true, all_pred, labels=labels)
pd.DataFrame(prediction_rows).sort_values(['subject_id', 'source_file']).to_csv(OUTPUT_DIR / 'predictions.csv', index=False)
pd.DataFrame(matrix, index=labels, columns=labels).to_csv(OUTPUT_DIR / 'confusion_matrix.csv')

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(matrix, cmap='Blues')
fig.colorbar(image, ax=ax)
ax.set(xticks=range(len(labels)), yticks=range(len(labels)), xticklabels=labels, yticklabels=labels, xlabel='Predicted', ylabel='True', title=f'Confusion matrix: {best_name}')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, matrix[i, j], ha='center', va='center')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150)
plt.close(fig)

best_model.fit(X, y)
joblib.dump({'model': best_model, 'feature_columns': feature_columns, 'classes': labels}, OUTPUT_DIR / 'best_model.joblib')

print('Best model:', best_name)
print('Confusion matrix:')
print(pd.DataFrame(matrix, index=labels, columns=labels).to_string())


Best model: LogisticRegression
Confusion matrix:
               antalgic  lurch  normal  steppage  stiff_legged  trendelenburg
antalgic            168      0      25        23           118             32
lurch                 0    318       0         0            21             20
normal               28      0     310        10             1             15
steppage              2      0      26       341             0              1
stiff_legged         63      1      13         2           276             11
trendelenburg        14     48      30         8            30            227


## 5) Predict a single walk

The saved model can now be reused on any row from the feature table, which is useful for one-off classification checks or deployment-style inference.

In [14]:
# Part 6: Inference on one walk
def predict_one_walk(row_index=0):
    table = pd.read_csv(FEATURES_PATH)
    if row_index < 0 or row_index >= len(table):
        raise IndexError(f'row_index must be between 0 and {len(table) - 1}')

    bundle = joblib.load(OUTPUT_DIR / 'best_model.joblib')
    row = table.iloc[[row_index]]
    probabilities = bundle['model'].predict_proba(row[bundle['feature_columns']])[0]
    ranked = sorted(zip(bundle['model'].classes_, probabilities), key=lambda item: item[1], reverse=True)

    print('Row:', row_index, 'source_file:', row.iloc[0]['source_file'])
    print('Predicted gait:', ranked[0][0])
    for label, probability in ranked:
        print(f'  {label}: {probability:.4f}')
    return ranked

predict_one_walk(0)


Row: 0 source_file: human1_antalgic1/human1_antalgic1_SkeletonData1.csv
Predicted gait: antalgic
  antalgic: 0.9884
  stiff_legged: 0.0113
  normal: 0.0002
  trendelenburg: 0.0001
  steppage: 0.0000
  lurch: 0.0000


[('antalgic', np.float64(0.9883814997813603)),
 ('stiff_legged', np.float64(0.01134271041516685)),
 ('normal', np.float64(0.00021287158350825878)),
 ('trendelenburg', np.float64(6.277902081679026e-05)),
 ('steppage', np.float64(1.1969127287306086e-07)),
 ('lurch', np.float64(1.950787486563724e-08))]

## 6) Live demo: predict a walk from a held-out subject

The model is trained without the demo subject, so the prediction is on a walk it has never seen.

In [15]:
# Demo: predict a walk from a test subject the model never saw in training
demo_model = make_models()[best_name]
demo_model.fit(X, y)
walk = test_features.iloc[[0]]
probs = demo_model.predict_proba(walk[feature_columns])[0]
ranked = sorted(zip(demo_model.classes_, probs), key=lambda p: p[1], reverse=True)
print("Test subject:", walk.iloc[0]["subject_id"])
print("Walk file:", walk.iloc[0]["source_file"])
print("True gait:", walk.iloc[0]["gait_label"])
print("Predicted gait:", ranked[0][0])
for label, p in ranked:
    print(f"  {label:15s} {p:.2f}")


Test subject: 8
Walk file: human8_antalgic1/human8_antalgic1_SkeletonData1.csv
True gait: antalgic
Predicted gait: normal
  normal          0.41
  stiff_legged    0.36
  antalgic        0.23
  trendelenburg   0.00
  lurch           0.00
  steppage        0.00
